# Movie details from TMDB (plot, director, cast, poster)

ML-1M's movies were matched to Wikipedia plots; for the recent movies we use
[TMDB](https://www.themoviedb.org/) instead - MovieLens 32M already links every movie to its TMDB id, so there is
no title matching. For each of the 4,000 movies this adds to `data/ml-32m-recent/movies_wiki.csv`:

| column | from TMDB | used for |
|---|---|---|
| `Plot` | the overview | Qwen's `plot` and `identify` training tasks |
| `Director`, `Cast` | the credits (top 5 cast) | Qwen's `details` task |
| `poster_path` | the poster | the UI |

Needs a TMDB API key in the environment: `TMDB_API_KEY=... jupyter ...` (free, themoviedb.org → Settings → API).
Re-running only fetches the movies that are still missing.

*This product uses the TMDB API but is not endorsed or certified by TMDB.*

In [1]:
import json
import os
import time
import urllib.error
import urllib.request
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

from genrec.data import load_movies
from genrec.paths import DATA_DIR

PATH = DATA_DIR / "ml-32m-recent" / "movies_wiki.csv"
KEY = os.environ.get("TMDB_API_KEY")
assert KEY, "set TMDB_API_KEY"
movies = pd.read_csv(PATH)
for col in ["Plot", "Director", "Cast", "poster_path"]:
    if col not in movies:
        movies[col] = None
todo = movies.index[movies["Plot"].isna() & movies["tmdb_id"].notna()]
print(f"{len(movies)} movies, {len(todo)} to fetch")

4000 movies, 4000 to fetch


In [2]:
def details(tmdb_id: int) -> dict:
    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}?append_to_response=credits&api_key={KEY}"
    for attempt in range(5):
        try:
            with urllib.request.urlopen(url, timeout=20) as r:
                d = json.load(r)
            directors = [c["name"] for c in d.get("credits", {}).get("crew", []) if c.get("job") == "Director"]
            cast = [c["name"] for c in d.get("credits", {}).get("cast", [])[:5]]
            return {"Plot": d.get("overview") or None, "Director": ", ".join(directors) or None,
                    "Cast": ", ".join(cast) or None, "poster_path": d.get("poster_path")}
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return {}
            time.sleep(float(e.headers.get("Retry-After", 1 + attempt)) if e.code == 429 else 1 + attempt)
        except urllib.error.URLError:
            time.sleep(1 + attempt)
    return {}


t0 = time.time()
with ThreadPoolExecutor(8) as pool:
    found = list(pool.map(lambda i: details(int(movies.at[i, "tmdb_id"])), todo))
for i, d in zip(todo, found):
    for col, value in d.items():
        movies.at[i, col] = value
movies.to_csv(PATH, index=False)
print(f"fetched {len(todo)} in {time.time() - t0:.0f}s -> {PATH}")

fetched 4000 in 452s -> data/ml-32m-recent/movies_wiki.csv


## Coverage

In [3]:
movies = load_movies(PATH.parent)        # what the training code reads
movies[["Plot", "Director", "Cast", "poster_path"]].notna().mean().to_frame("movies with it").style.format("{:.1%}")

,movies with it
Plot,98.9%
Director,98.8%
Cast,98.7%
poster_path,98.9%


In [4]:
movies[["display", "Director", "Cast", "Plot"]].sample(5, random_state=0)

,display,Director,Cast,Plot
2230,Sausage Party (2016),"Greg Tiernan, Conrad Vernon","Seth Rogen, Kristen Wiig, Jonah Hill, Bill Had...",Hot dog Frank leads a group of supermarket pro...
668,Your Sister's Sister (2011),Lynn Shelton,"Emily Blunt, Rosemarie DeWitt, Mark Duplass, M...",Iris invites her friend Jack to stay at her fa...
3616,Demon Slayer the Movie: Mugen Train (2020),Haruo Sotozaki,"Natsuki Hanae, Akari Kito, Hiro Shimono, Yoshi...","Tanjiro Kamado, joined with Inosuke Hashibira,..."
2363,All Eyez on Me (2016),Benny Boom,"Demetrius Shipp Jr., Danai Gurira, Kat Graham,...",All Eyez on Me chronicles the life and legacy ...
142,Restrepo (2010),"Tim Hetherington, Sebastian Junger","Juan ""Doc"" Restrepo, Dan Kearney, LaMonta Cald...",Directors Hetherington and Junger spend a year...
